In [0]:
# ============================================================
# NOTEBOOK 1 — DataPrep | Case Técnico iFood
# Autoria: Ana Clara Aragão Fernandes
# ============================================================
# Objetivo: 
#   1. Carregar as bases de dados transactions.json, customers.json e offers.json
#   2. Feature engineering da base de dados
#   3. Limpeza e validação dos dados
#   4. Join das bases usando a lógica de funil
#
# Input:  
# Output: workspace.default.ifood_final_dataset (76.277 linhas, 27 colunas)
# ============================================================

In [0]:

import os
os.listdir("/Volumes/workspace/default/ifood_case")
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.sql.window import Window

VOLUME_PATH = "/Volumes/workspace/default/ifood_case"

### Carga dos dados e testes iniciais

**Nota sobre obtenção dos dados:**
O bucket S3 público referenciado no enunciado do case (`ds-technical-evaluation-source`) estava indisponível no momento da execução.
(erro `NoSuchBucket` confirmado tanto via notebook quanto via acesso direto pelo navegador, descartando bloqueio de rede do ambiente Databricks). 
Após contato com a equipe do iFood, os dados foram reenviados como arquivo .zip e carregados manualmente via Unity Catalog Volume. 
Por essa razão, os dados brutos (`offers.json`, `profile.json`, `transactions.json`) estão versionados em `data/raw/` neste repositório, garantindo reprodutibilidade mesmo que o bucket original permaneça indisponível.

In [0]:
import shutil
import os

GIT_FOLDER_DATA_RAW = "/Workspace/Users/13aclara@gmail.com/ifood-case202606/data/raw" 
VOLUME_PATH = "/Volumes/workspace/default/ifood_case"

for f in ["offers.json", "profile.json", "transactions.json"]:
    origem = f"{VOLUME_PATH}/{f}"
    destino = f"{GIT_FOLDER_DATA_RAW}/{f}"
    shutil.copy(origem, destino)
    print(f"Copiado: {f}")

print("\nConteúdo da pasta data/raw no Git folder:")
print(os.listdir(GIT_FOLDER_DATA_RAW))

#leitura dos arquivos JSON com os dados

In [0]:
transactions_df = spark.read.json(f"{VOLUME_PATH}/transactions.json")
offers_df = spark.read.json(f"{VOLUME_PATH}/offers.json")
customers_df = spark.read.json(f"{VOLUME_PATH}/profile.json")



In [0]:

print("=== TRANSACTIONS ===")
transactions_df.printSchema()
transactions_df.show(5, truncate=False)

In [0]:

print("=== OFFERS ===")
offers_df.printSchema()
offers_df.show(5, truncate=False)



In [0]:
print("=== CUSTOMERS (profile.json) ===")
customers_df.printSchema()
customers_df.show(5, truncate=False)


# 1. Transactions

In [0]:
#Check de eventos disponiveis:
transactions_df.select("event").distinct().show(truncate=False)


In [0]:
for ev in [r["event"] for r in transactions_df.select("event").distinct().collect()]:
    print(f"\n--- event = '{ev}' ---")
    transactions_df.filter(F.col("event") == ev).select("event", "value").show(10, truncate=False)

temos 4 tipos de eventos:
* offer received - cliente recebe a oferta (campo `offer id` do vetor de valores)
* offer viewed - cliente visualiza a oferta (campo `offer id` do vetor de valores)
* offer completed -> cliente recebe o cupom - inclui o valor do cupom (campos `offer_id`, `reward` do vetor de valores)
* transaction -> compra efetivada (campo `amount` do vetor de valores)

-> campos do vetor de valores: [`amount`, `offer id`, `offer_id`, `reward`]

**Duplicação do campo offer id -> provavelmente algum débito tecnico na geração do arquivo; nao impacta no modelo mas exige a unificação dos campos no tratamento da base**


In [0]:
transactions_clean = (
    transactions_df
    .withColumnRenamed("account_id", "customer_id")
    .withColumn("offer_id", F.coalesce(F.col("value.`offer id`"), F.col("value.offer_id")))
    .withColumn("amount", F.col("value.amount"))
    .withColumn("reward", F.col("value.reward"))
    .drop("value")
)

transactions_clean.show(5, truncate=False)
print("Total de linhas:", transactions_clean.count())

In [0]:
transactions_clean = transactions_clean.withColumn(
    "event_clean",
    F.when(F.col("event") == "offer received", "offer_received")
     .when(F.col("event") == "offer viewed", "offer_viewed")
     .when(F.col("event") == "offer completed", "offer_completed")
     .when(F.col("event") == "transaction", "transaction")
     .otherwise(F.col("event"))
)

transactions_df.groupBy("event").count().show()
transactions_clean.groupBy("event_clean").count().show()



In [0]:
transactions_clean.show()

## Verifica se há duplicação na base transacional

In [0]:
#check
total_linhas = transactions_clean.count()
linhas_distintas = transactions_clean.dropDuplicates().count()

print(f"Total de linhas: {total_linhas}")
print(f"Linhas distintas (todas as colunas): {linhas_distintas}")
print(f"Duplicatas exatas: {total_linhas - linhas_distintas}")
print(f"% duplicação: {(total_linhas - linhas_distintas)/total_linhas}")

In [0]:
# Identifica as combinações exatas que se repetem (linha 100% idêntica em todas as colunas)
dup_exatas = (
    transactions_clean
    .groupBy(transactions_clean.columns)
    .count()
    .filter(F.col("count") > 1)
)

print("Distribuição das duplicatas por tipo de evento:")
dup_exatas.groupBy("event_clean").agg(
    F.sum("count").alias("total_linhas_envolvidas"),
    F.count("*").alias("combinacoes_unicas_duplicadas")
).show()

In [0]:
exemplo_dup = (
    transactions_clean
    .groupBy("customer_id", "event_clean", "time_since_test_start", "offer_id", "reward")
    .count()
    .filter((F.col("event_clean") == "offer_completed") & (F.col("count") > 1))
    .limit(5)
)

exemplo_dup.show(truncate=False)

# Pega um customer_id de exemplo e mostra a timeline completa dele
cliente_exemplo_dup = exemplo_dup.collect()[0]["customer_id"]
transactions_clean.filter(F.col("customer_id") == cliente_exemplo_dup) \
    .orderBy("time_since_test_start") \
    .select("customer_id", "event_clean", "time_since_test_start", "offer_id", "reward","amount") \
    .show(50, truncate=False)

### 🔴 Hipótese de concessão duplicada de cupom 

com os dados disponibilizados do case, não consigo validar se é de fato concessão duplicada ou apenas uma quetão de log de mensageria duplicado (por fila ou outro motivo) - em operação de dia a dia, vale um double check na base e acompanhamento com time responsável




In [0]:

transactions_clean = transactions_clean.withColumn("_row_id", F.monotonically_increasing_id())

w_dedup = Window.partitionBy(*[c for c in transactions_clean.columns if c != "_row_id"]).orderBy("_row_id")

transactions_with_rank = transactions_clean.withColumn("_dup_rank", F.row_number().over(w_dedup))

# Base limpa: só a primeira ocorrência de cada transação
transactions_dedup = (
    transactions_with_rank
    .filter(F.col("_dup_rank") == 1)
    .drop("_row_id", "_dup_rank")
)

# Base de duplicatas removidas
transactions_duplicates_removed = (
    transactions_with_rank
    .filter(F.col("_dup_rank") > 1)
    .drop("_row_id", "_dup_rank")
)

print("Linhas na base limpa:", transactions_dedup.count())
print("Linhas removidas como duplicata:", transactions_duplicates_removed.count())

In [0]:
(
    transactions_dedup
    .write
    .mode("overwrite")
    .saveAsTable("workspace.default.transactions_clean")
)

(
    transactions_duplicates_removed
    .write
    .mode("overwrite")
    .saveAsTable("workspace.default.transactions_duplicates_removed")
)

print("Tabelas salvas")

#check de volumes
spark.sql("SELECT COUNT(*) FROM workspace.default.transactions_clean").show()
spark.sql("SELECT COUNT(*) FROM workspace.default.transactions_duplicates_removed").show()

A base de duplicatas foi persistida para posterior verificação de concessão duplicada de cumpom (Versus falha de mensageria): Na amostra, o volume é pequeno, mas na base toda, podemos estar falando de um custo operacional significativo.

# 2. Offers

### Base de ofertas - tratamentos realizados
Neste case, a base é reduzida, contém apenas 10 ofertas. Foi validada visualmente e os tratamentos foram apenas para facilitação na etapa de modelagem, além de verificações padrão:
1. Validação de distintos
2. Canais de envio: ['email', 'mobile', 'web', 'social']
3. Tipos de Oferta ['BoGo', 'discount', 'informational']
4. Describe de variáveis numéricas
5. Variáveis dummy de canais (para ter a opção de usar na etapa de modelagem)

In [0]:
canais_distintos = [
    row["canal"] for row in
    offers_df.select(F.explode("channels").alias("canal")).distinct().collect()
]
print("Canais distintos:", canais_distintos)

In [0]:
offers_df.show()

In [0]:
print("Total de ofertas:", offers_df.count())
print("IDs distintos:", offers_df.select("id").distinct().count())
print("Linhas distintas (todas as colunas):", offers_df.dropDuplicates().count())

In [0]:
offers_df.groupBy("offer_type").count().orderBy(F.desc("count")).show()

In [0]:
offers_df.select("min_value", "duration", "discount_value").describe().show()

In [0]:
offers_df.groupBy("offer_type").agg(
    F.min("min_value").alias("min_value_min"),
    F.max("min_value").alias("min_value_max"),
    F.min("duration").alias("duration_min"),
    F.max("duration").alias("duration_max"),
    F.min("discount_value").alias("discount_min"),
    F.max("discount_value").alias("discount_max"),
).show()

In [0]:
offers_df.orderBy("offer_type").show(50, truncate=False)

In [0]:
# Dummies de canal
canais_distintos = [
    row["canal"] for row in
    offers_df.select(F.explode("channels").alias("canal")).distinct().collect()
]
print("Canais distintos:", canais_distintos)

offers_clean = offers_df.withColumnRenamed("id", "offer_id")

for canal in canais_distintos:
    offers_clean = offers_clean.withColumn(
        f"channel_{canal}",
        F.array_contains(F.col("channels"), canal).cast("int")
    )

offers_clean.show(truncate=False)
offers_clean.printSchema()

In [0]:
(
    offers_clean
    .write
    .mode("overwrite")
    .saveAsTable("workspace.default.offers_clean")
)

print("Tabela salva: workspace.default.offers_clean")
spark.sql("SELECT COUNT(*) FROM workspace.default.offers_clean").show()

# 3. Clientes

### Base de Clientes - tratamentos realizados
Com dados de clientes, a validação é cadastral e informacional:
1. Check de Idades 

🔴 Idade máxima = 118 -> aparente SENTINELA de dados (valor fictício que indica ausencia de informação ou dado de tratamento especial)


In [0]:
customers_df.select(
    F.min("age").alias("idade_min"),
    F.max("age").alias("idade_max"),
    F.mean("age").alias("idade_media"),
    F.expr("percentile_approx(age, 0.5)").alias("idade_mediana")
).show()

In [0]:
customers_df.groupBy("age").count().orderBy(F.desc("age")).show(20)



In [0]:
import matplotlib.pyplot as plt

# Traz só a coluna de idade pra pandas (leve, não precisa do resto)
idades_pd = customers_df.select("age").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Com o sentinela (118 incluso)
axes[0].hist(idades_pd["age"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Distribuição de idade — COM sentinela (118)")
axes[0].set_xlabel("Idade")
axes[0].set_ylabel("Frequência")

# Sem o sentinela (118 removido)
idades_sem_sentinela = idades_pd[idades_pd["age"] != 118]
axes[1].hist(idades_sem_sentinela["age"], bins=50, color="seagreen", edgecolor="white")
axes[1].set_title("Distribuição de idade — SEM sentinela (118 removido)")
axes[1].set_xlabel("Idade")
axes[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

#### Analise de idade - confirma suspeita de age=118 como sentinela
garantia de que o a idade=118 é um sentinela - para fins de modelagem, criação de flag de idade 118 + substituição por NULL na coluna original

obs: em 100% dos casos, coincide com gender =null e limite de credito = null

In [0]:
print("Total de clientes:", customers_df.count())
print("IDs distintos:", customers_df.select("id").distinct().count())

print("\nClientes com age = 118:")
customers_df.filter(F.col("age") == 118).count()

In [0]:
# Confirma se TODO registro com age=118 tem gender e credit_card_limit nulos
customers_df.filter(F.col("age") == 118).select(
    F.count("*").alias("total"),
    F.count("gender").alias("gender_nao_nulo"),
    F.count("credit_card_limit").alias("limit_nao_nulo")
).show()

In [0]:
customers_clean = (
    customers_df
    .withColumn("age_missing", (F.col("age") == 118).cast("int"))
    .withColumn("age", F.when(F.col("age") == 118, None).otherwise(F.col("age")))
)

# Confirma o resultado
customers_clean.select(
    F.count("*").alias("total"),
    F.count("age").alias("age_nao_nulo"),
    F.sum("age_missing").alias("total_flagged_missing")
).show()

In [0]:
customers_clean.groupBy("age_missing").agg(
    F.count("*").alias("total"),
    F.count("gender").alias("gender_preenchido"),
    F.count("credit_card_limit").alias("limit_preenchido")
).show()

In [0]:
customers_clean = (
    customers_clean
    .withColumnRenamed("id", "customer_id")
    .withColumn(
        "registered_on_date",
        F.to_date(F.col("registered_on").cast("string"), "yyyyMMdd")
    )
)

customers_clean.select("customer_id", "age", "age_missing", "gender", "credit_card_limit", "registered_on", "registered_on_date").show(10, truncate=False)

In [0]:
(
    customers_clean
    .write
    .mode("overwrite")
    .saveAsTable("workspace.default.customers_clean")
)

print("Tabela salva: workspace.default.customers_clean")
spark.sql("SELECT COUNT(*) FROM workspace.default.customers_clean").show()

# PARTE 2: Funil de conversão

In [0]:
transactions_clean = spark.table("workspace.default.transactions_clean")
offers_clean = spark.table("workspace.default.offers_clean")


In [0]:
# time_since_test_start: range observado
transactions_clean.select(
    F.min("time_since_test_start").alias("tempo_min"),
    F.max("time_since_test_start").alias("tempo_max")
).show()

# duration: range observado
offers_clean.select(
    F.min("duration").alias("duration_min"),
    F.max("duration").alias("duration_max")
).show()

In [0]:
from pyspark.sql.window import Window

received = transactions_clean.filter(F.col("event_clean") == "offer_received")
viewed = transactions_clean.filter(F.col("event_clean") == "offer_viewed")
completed = transactions_clean.filter(F.col("event_clean") == "offer_completed")

# Sequência de instância: se o mesmo cliente recebe a mesma oferta mais de uma vez,
# cada recebimento vira uma instância separada e independente do funil
w_seq = Window.partitionBy("customer_id", "offer_id").orderBy("time_since_test_start")
received = received.withColumn("offer_instance_seq", F.row_number().over(w_seq))

# Traz duration e offer_type da tabela de ofertas, pra delimitar a janela de validade
received = (
    received
    .join(
        offers_clean.select("offer_id", "duration", "offer_type"),
        on="offer_id",
        how="left"
    )
    .withColumnRenamed("time_since_test_start", "received_time")
)

received.select(
    "customer_id", "offer_id", "offer_instance_seq", "received_time", "duration", "offer_type"
).show(10, truncate=False)

In [0]:
received.groupBy("customer_id", "offer_id").count().filter(F.col("count") > 1).count()

In [0]:
received.groupBy("offer_id").agg(
    F.min("received_time").alias("primeiro_recebimento"),
    F.max("received_time").alias("ultimo_recebimento"),
    F.countDistinct("received_time").alias("dias_distintos_de_envio"),
    F.count("*").alias("total_recebimentos")
).orderBy("offer_id").show(20, truncate=False)

In [0]:
transaction_events = transactions_clean.filter(F.col("event_clean") == "transaction")

# Pra cada transação, verifica se existe ALGUMA oferta recebida (de qualquer tipo)
# pelo mesmo cliente, cuja janela [received_time, received_time+duration] contenha o tempo da transação
transacoes_com_oferta_ativa = transaction_events.alias("t").join(
    received.alias("r"),
    on=[
        F.col("t.customer_id") == F.col("r.customer_id"),
        F.col("t.time_since_test_start") >= F.col("r.received_time"),
        F.col("t.time_since_test_start") <= F.col("r.received_time") + F.col("r.duration"),
    ],
    how="left"
)

total_transacoes = transaction_events.count()
transacoes_com_match = transacoes_com_oferta_ativa.filter(F.col("r.customer_id").isNotNull()).select("t.customer_id", "t.time_since_test_start").distinct().count()

print("Total de eventos de transação:", total_transacoes)
print("Transações que caem dentro de alguma janela de oferta ativa:", transacoes_com_match)
print("Transações SEM nenhuma oferta ativa no momento (potencial baseline):", total_transacoes - transacoes_com_match)

In [0]:
# Join: view dentro da janela de validade [received_time, received_time + duration]
viewed_join = received.alias("r").join(
    viewed.alias("v"),
    on=[
        F.col("r.customer_id") == F.col("v.customer_id"),
        F.col("v.offer_id") == F.col("r.offer_id"),
        F.col("v.time_since_test_start") >= F.col("r.received_time"),
        F.col("v.time_since_test_start") <= F.col("r.received_time") + F.col("r.duration"),
    ],
    how="left"
)

# Pega o view mais próximo (cronologicamente) de cada instância de recebimento,
# pra não duplicar quando existir mais de um view dentro da mesma janela
w_nearest_view = Window.partitionBy(
    "r.customer_id", "r.offer_id", "r.offer_instance_seq"
).orderBy(F.col("v.time_since_test_start").asc_nulls_last())

viewed_join = (
    viewed_join
    .withColumn("rn", F.row_number().over(w_nearest_view))
    .filter(F.col("rn") == 1)
    .select(
        F.col("r.customer_id").alias("customer_id"),
        F.col("r.offer_id").alias("offer_id"),
        F.col("r.offer_instance_seq").alias("offer_instance_seq"),
        F.col("r.received_time").alias("received_time"),
        F.col("r.duration").alias("duration"),
        F.col("r.offer_type").alias("offer_type"),
        F.col("v.time_since_test_start").alias("viewed_time"),
    )
)

viewed_join.show(20, truncate=False)
print("Total de linhas:", viewed_join.count())
print("Com view dentro da janela:", viewed_join.filter(F.col("viewed_time").isNotNull()).count())

In [0]:
# Join: completed dentro da janela de validade, e DEPOIS do view (quando houver view)
completed_join = viewed_join.alias("vj").join(
    completed.alias("c"),
    on=[
        F.col("vj.customer_id") == F.col("c.customer_id"),
        F.col("vj.offer_id") == F.col("c.offer_id"),
        F.col("c.time_since_test_start") >= F.col("vj.received_time"),
        F.col("c.time_since_test_start") <= F.col("vj.received_time") + F.col("vj.duration"),
    ],
    how="left"
)

w_nearest_complete = Window.partitionBy(
    "vj.customer_id", "vj.offer_id", "vj.offer_instance_seq"
).orderBy(F.col("c.time_since_test_start").asc_nulls_last())

funnel = (
    completed_join
    .withColumn("rn", F.row_number().over(w_nearest_complete))
    .filter(F.col("rn") == 1)
    .select(
        F.col("vj.customer_id").alias("customer_id"),
        F.col("vj.offer_id").alias("offer_id"),
        F.col("vj.offer_instance_seq").alias("offer_instance_seq"),
        F.col("vj.received_time").alias("received_time"),
        F.col("vj.duration").alias("duration"),
        F.col("vj.offer_type").alias("offer_type"),
        F.col("vj.viewed_time").alias("viewed_time"),
        F.col("c.time_since_test_start").alias("completed_time"),
        F.col("c.reward").alias("reward"),
    )
)

funnel = (
    funnel
    .withColumn("was_viewed", F.col("viewed_time").isNotNull().cast("int"))
    .withColumn("was_completed", F.col("completed_time").isNotNull().cast("int"))
    .withColumn(
        "target_engaged_conversion",
        F.when((F.col("was_viewed") == 1) & (F.col("was_completed") == 1), 1).otherwise(0)
    )
)

funnel.show(20, truncate=False)
print("Total de linhas:", funnel.count())
print("Visto:", funnel.filter(F.col("was_viewed")==1).count())
print("Completado:", funnel.filter(F.col("was_completed")==1).count())
print("Engajamento real (visto E completado):", funnel.filter(F.col("target_engaged_conversion")==1).count())
funnel.groupBy("offer_type", "target_engaged_conversion").count().orderBy("offer_type").show()

##### Necessidade de definição prática para ofertas informacionais: definição de sucesso = Transação concluida no período da janela de validade


In [0]:
# Join: transações dentro da janela de validade, e DEPOIS do view (mantém a mesma lógica causal)
transaction_join = viewed_join.alias("vj").join(
    transaction_events.alias("t"),
    on=[
        F.col("vj.customer_id") == F.col("t.customer_id"),
        F.col("t.time_since_test_start") >= F.col("vj.received_time"),
        F.col("t.time_since_test_start") <= F.col("vj.received_time") + F.col("vj.duration"),
    ],
    how="left"
)

# Pega a transação mais próxima de cada instância
w_nearest_transaction = Window.partitionBy(
    "vj.customer_id", "vj.offer_id", "vj.offer_instance_seq"
).orderBy(F.col("t.time_since_test_start").asc_nulls_last())

transaction_join = (
    transaction_join
    .withColumn("rn", F.row_number().over(w_nearest_transaction))
    .filter(F.col("rn") == 1)
    .select(
        F.col("vj.customer_id").alias("customer_id"),
        F.col("vj.offer_id").alias("offer_id"),
        F.col("vj.offer_instance_seq").alias("offer_instance_seq"),
        F.col("t.time_since_test_start").alias("transaction_time"),
        F.col("t.amount").alias("transaction_amount"),
    )
)

In [0]:
transaction_events = transactions_clean.filter(F.col("event_clean") == "transaction")

In [0]:
funnel = funnel.join(
    transaction_join.select("customer_id", "offer_id", "offer_instance_seq", "transaction_time", "transaction_amount"),
    on=["customer_id", "offer_id", "offer_instance_seq"],
    how="left"
)

funnel = funnel.withColumn(
    "had_transaction_in_window", F.col("transaction_time").isNotNull().cast("int")
)

# Critério de sucesso condicional por tipo de oferta
funnel = funnel.withColumn(
    "target_engaged_conversion",
    F.when(
        F.col("offer_type") == "informational",
        ((F.col("was_viewed") == 1) & (F.col("had_transaction_in_window") == 1)).cast("int")
    ).otherwise(
        ((F.col("was_viewed") == 1) & (F.col("was_completed") == 1)).cast("int")
    )
)

funnel.groupBy("offer_type", "target_engaged_conversion").count().orderBy("offer_type").show()

In [0]:
funnel_com_perfil = funnel.join(
    customers_clean.select("customer_id", "age", "age_missing", "gender", "credit_card_limit"),
    on="customer_id",
    how="left"
)

In [0]:
funnel_com_perfil = funnel_com_perfil.withColumn(
    "faixa_etaria",
    F.when(F.col("age_missing") == 1, "desconhecida")
     .when(F.col("age") < 25, "<25")
     .when(F.col("age") < 35, "25-34")
     .when(F.col("age") < 45, "35-44")
     .when(F.col("age") < 55, "45-54")
     .when(F.col("age") < 65, "55-64")
     .otherwise("65+")
)

funnel_com_perfil.groupBy("faixa_etaria").agg(
    F.count("*").alias("total"),
    F.avg("target_engaged_conversion").alias("taxa_conversao")
).orderBy("faixa_etaria").show()

In [0]:
customers_clean = spark.table("workspace.default.customers_clean")
offers_clean = spark.table("workspace.default.offers_clean")

final_df = (
    funnel
    .join(
        customers_clean,
        on="customer_id",
        how="left"
    )
    .join(
        offers_clean
        .drop("offer_type")   # já está no funnel
        .drop("duration")     # já está no funnel
        .drop("channels"),    # já tem os dummies, array não entra no modelo
        on="offer_id",
        how="left"
    )
)

print("Total de linhas:", final_df.count())
print("Total de colunas:", len(final_df.columns))
final_df.printSchema()

In [0]:
# Checar nulos nas colunas críticas de join
final_df.select(
    F.count(F.when(F.col("min_value").isNull(), 1)).alias("min_value_null"),
    F.count(F.when(F.col("discount_value").isNull(), 1)).alias("discount_value_null"),
    F.count(F.when(F.col("channel_email").isNull(), 1)).alias("channel_email_null"),
    F.count(F.when(F.col("age").isNull() & (F.col("age_missing") == 0), 1)).alias("age_null_inesperado"),
).show()

In [0]:
(
    final_df
    .write
    .mode("overwrite")
    .saveAsTable("workspace.default.ifood_final_dataset")
)

print("Tabela salva: workspace.default.ifood_final_dataset")
spark.sql("SELECT COUNT(*) FROM workspace.default.ifood_final_dataset").show()
spark.sql("SELECT offer_type, AVG(target_engaged_conversion) as taxa_conversao, COUNT(*) as total FROM workspace.default.ifood_final_dataset GROUP BY offer_type").show()